In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType
)
import time
import requests
from datetime import datetime
from pyspark.sql.functions import col, countDistinct

spark = SparkSession.builder \
    .master("spark://group13-master:7077") \
    .appName("Parquet-conversion") \
    .config("spark.dynamicAllocation.enabled", "true") \
    .config("spark.shuffle.service.enabled", "false") \
    .config("spark.dynamicAllocation.shuffleTracking.enabled", "true") \
    .enableHiveSupport() \
    .getOrCreate()

26/03/19 13:04:34 WARN StandaloneSchedulerBackend: Dynamic allocation enabled without spark.executor.cores explicitly set, you may get more executors allocated than expected. It's recommended to set spark.executor.cores explicitly. Please check SPARK-30299 for more details.


In [8]:
reddit_schema = StructType([
    StructField("author", StringType(), True),
    StructField("body", StringType(), True),
    StructField("normalizedBody", StringType(), True),
    StructField("content", StringType(), True),
    StructField("content_len", LongType(), True),
    StructField("summary", StringType(), True),
    StructField("summary_len", LongType(), True),
    StructField("id", StringType(), True),
    StructField("subreddit", StringType(), True),
    StructField("subreddit_id", StringType(), True),
    StructField("title", StringType(), True)
])

path = "hdfs://group13-master:9000/project/reddit/raw"
parquet_path = "hdfs:///project/reddit/parquet/part1"

df = spark.read.schema(reddit_schema).json(path)
df.show(5)

[Stage 0:>                                                          (0 + 1) / 1]

+---------------+--------------------+--------------------+--------------------+-----------+--------------------+-----------+-------+---------+------------+-----+
|         author|                body|      normalizedBody|             content|content_len|             summary|summary_len|     id|subreddit|subreddit_id|title|
+---------------+--------------------+--------------------+--------------------+-----------+--------------------+-----------+-------+---------+------------+-----+
|        MHTLuca|Probably one of m...|Probably one of m...|Probably one of m...|        223|that shit...prett...|         12|c1tw02f|AskReddit|    t5_2qh1i| NULL|
|soxandpatriots1|If you start now,...|If you start now,...|If you start now,...|        300|Do a few days of ...|         16|c1tzgbh|  running|    t5_2qlit| NULL|
|     Fu_Man_Chu|I went to Dayton ...|I went to Dayton ...|I went to Dayton ...|        129|Dayton seems like...|         12|c1u8kh4|     pics|    t5_2qh0u| NULL|
|        afkmofo|Looks

In [9]:
df.write.parquet(parquet_path)

In [10]:
spark.stop()